In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchFrameException, NoSuchElementException
import time

# ============================================================================
# CONFIGURACIÓN (Mantener como está)
# ============================================================================

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-notifications")

ruta_driver = r"D:\Users\Lenovo\Documents\chrome-win\chromedriver.exe" 
service = Service(ruta_driver)
driver = webdriver.Chrome(service=service, options=chrome_options)

# ============================================================================
# CREDENCIALES Y VARIABLES (Mantener como está)
# ============================================================================

URL = "http://senasofiaplus.edu.co/sofia-public/"
USUARIO = "1050962935" 
CONTRASENA = "PapaJose92805331050*" 
FICHA_A_INGRESAR = "123456"

# IDs de iframes
IFRAME_LOGIN = 'registradoBox1'
IFRAME_CONTENIDO = 'contenido'
IFRAME_MODAL = 'modalDialogContentviewDialog'  

# XPaths
XPATH_MENU_MATRICULA = '//*[@id="side-menu"]/li[5]/a'
XPATH_OPCION_PROCESO_MATRICULA = '//*[@id="side-menu"]/li[5]/ul/li[2]/a'
XPATH_FINALIZAR_PROCESO = '//*[@id="3115Opcion"]' 

# FILTROS INICIALES DENTRO DEL IFRAME 'contenido'
ID_SELECT_TIPO_FORMACION = 'tipoFormacion'
XPATH_BOTON_BUSCAR_FICHA = '//*[@id="frmFichas:buscarFichas"]'

# MODAL DE FILTROS POSTERIOR
XPATH_ICONO_FILTROS = '/html/body[1]/div[2]/div[1]/fieldset/form/table/tbody/tr/td[3]/a/img'
XPATH_INPUT_FICHA = '/html/body/div[2]/form/fieldset/div/table/tbody/tr[1]/td[2]/input'
XPATH_BOTON_BUSCAR_MODAL = '//*[@id="form:buscarCBT"]' 

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def imprimir_seccion(titulo):
    """Imprime un título de sección formateado"""
    print("\n" + "="*70)
    print(f"  {titulo}")
    print("="*70)

def listar_todos_los_iframes():
    """Lista todos los iframes disponibles en la página actual"""
    driver.switch_to.default_content()
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    print(f"📊 Total de iframes encontrados: {len(iframes)}")
    
    for i, iframe in enumerate(iframes):
        iframe_id = iframe.get_attribute("id") or "SIN ID"
        iframe_name = iframe.get_attribute("name") or "SIN NAME"
        iframe_src = iframe.get_attribute("src") or "SIN SRC"
        print(f"   [{i}] ID: '{iframe_id}' | NAME: '{iframe_name}' | SRC: '{iframe_src[:50]}...'")
    
    return len(iframes)

def buscar_cualquier_select():
    """Busca CUALQUIER elemento select en todos los iframes (para debugging)"""
    imprimir_seccion("🔍 BÚSQUEDA DE CUALQUIER <SELECT> (DEBUGGING)")
    
    driver.switch_to.default_content()
    num_iframes = len(driver.find_elements(By.TAG_NAME, "iframe"))
    
    for i in range(num_iframes):
        try:
            driver.switch_to.default_content()
            iframes_actuales = driver.find_elements(By.TAG_NAME, "iframe")
            
            if i >= len(iframes_actuales):
                continue
            
            iframe_id = iframes_actuales[i].get_attribute("id") or f"índice_{i}"
            driver.switch_to.frame(i)
            
            # Buscar TODOS los selects
            selects = driver.find_elements(By.TAG_NAME, "select")
            
            if selects:
                print(f"\n   ✅ Iframe [{i}] '{iframe_id}': {len(selects)} select(s) encontrado(s)")
                for j, sel in enumerate(selects):
                    sel_id = sel.get_attribute("id") or "SIN ID"
                    sel_name = sel.get_attribute("name") or "SIN NAME"
                    
                    try:
                        sel_obj = Select(sel)
                        opciones = [opt.text for opt in sel_obj.options]
                        print(f"      Select [{j}]: ID='{sel_id}', NAME='{sel_name}'")
                        print(f"         Opciones: {opciones[:5]}{'...' if len(opciones) > 5 else ''}")
                    except:
                        print(f"      Select [{j}]: ID='{sel_id}', NAME='{sel_name}' (error al leer opciones)")
                
                return True, i
            
        except Exception as e:
            continue
    
    print("\n❌ No se encontró NINGÚN select en NINGÚN iframe")
    return False, -1

def buscar_select_en_todos_los_iframes(select_id, select_opciones=None):
    """
    Busca un elemento <select> específico explorando todos los iframes
    
    Args:
        select_id: ID del elemento select a buscar
        select_opciones: Lista de textos de opciones esperadas (para verificar)
    
    Returns:
        tuple: (encontrado: bool, iframe_index: int, select_element: WebElement)
    """
    imprimir_seccion(f"🔍 BÚSQUEDA DE SELECT '{select_id}' EN TODOS LOS IFRAMES")
    
    driver.switch_to.default_content()
    # Contar iframes primero
    num_iframes = len(driver.find_elements(By.TAG_NAME, "iframe"))
    
    print(f"📊 Explorando {num_iframes} iframes...")
    
    # Iterar por ÍNDICE en lugar de por elemento para evitar stale references
    for i in range(num_iframes):
        try:
            # CRÍTICO: Volver a default_content y re-obtener el iframe cada vez
            driver.switch_to.default_content()
            
            # Obtener la lista actualizada de iframes
            iframes_actuales = driver.find_elements(By.TAG_NAME, "iframe")
            
            if i >= len(iframes_actuales):
                print(f"   ⚠️ Iframe [{i}] ya no existe")
                continue
            
            # Obtener el ID/nombre del iframe para logging
            iframe_actual = iframes_actuales[i]
            iframe_id = iframe_actual.get_attribute("id") or f"índice_{i}"
            iframe_name = iframe_actual.get_attribute("name") or "SIN NAME"
            
            print(f"\n   🔎 Explorando iframe [{i}]: ID='{iframe_id}', NAME='{iframe_name}'")
            
            # Cambiar al iframe por índice
            driver.switch_to.frame(i)
            
            # Intentar encontrar el select
            try:
                select_elem = driver.find_element(By.ID, select_id)
                print(f"   ✅ ¡SELECT ENCONTRADO en iframe [{i}]: '{iframe_id}'!")
                
                # Verificar opciones si se proporcionaron
                if select_opciones:
                    select_obj = Select(select_elem)
                    opciones_disponibles = [opt.text for opt in select_obj.options]
                    print(f"   📋 Opciones disponibles: {opciones_disponibles}")
                    
                    # Verificar si alguna opción esperada está presente
                    coincidencias = [opt for opt in select_opciones if opt in opciones_disponibles]
                    if coincidencias:
                        print(f"   ✅ Opciones esperadas encontradas: {coincidencias}")
                    else:
                        print(f"   ⚠️ Ninguna opción esperada encontrada")
                
                return True, i, select_elem
                
            except NoSuchElementException:
                print(f"   ❌ Select no encontrado en este iframe")
                
        except Exception as e:
            print(f"   ⚠️ Error al explorar iframe [{i}]: {str(e)[:80]}")
            continue
    
    driver.switch_to.default_content()
    print("\n❌ Select NO encontrado en ningún iframe")
    return False, -1, None

def seleccionar_opcion_en_select(select_element, texto_opciones):
    """
    Intenta seleccionar una opción del select probando diferentes variaciones
    
    Args:
        select_element: Elemento <select> de Selenium
        texto_opciones: Lista de variaciones del texto a buscar (ej: ["COMPLEMENTARIA", "Complementaria", "complementaria"])
    
    Returns:
        bool: True si se seleccionó exitosamente
    """
    select_obj = Select(select_element)
    opciones_disponibles = [opt.text for opt in select_obj.options]
    
    print(f"\n   📋 Opciones disponibles en el select:")
    for idx, opt_text in enumerate(opciones_disponibles):
        print(f"      [{idx}] '{opt_text}'")
    
    # Intentar por texto visible
    for texto in texto_opciones:
        try:
            select_obj.select_by_visible_text(texto)
            print(f"   ✅ Opción seleccionada: '{texto}'")
            return True
        except:
            print(f"   ⚠️ No se pudo seleccionar por texto: '{texto}'")
    
    # Intentar por coincidencia parcial
    for texto in texto_opciones:
        for opt in opciones_disponibles:
            if texto.upper() in opt.upper():
                try:
                    select_obj.select_by_visible_text(opt)
                    print(f"   ✅ Opción seleccionada por coincidencia: '{opt}'")
                    return True
                except:
                    pass
    
    # Último intento: seleccionar por índice (si sabemos que es la opción 2, índice 1)
    try:
        if len(opciones_disponibles) > 1:
            select_obj.select_by_index(1)
            print(f"   ✅ Opción seleccionada por índice [1]: '{opciones_disponibles[1]}'")
            return True
    except:
        pass
    
    print("   ❌ No se pudo seleccionar ninguna opción")
    return False

# ============================================================================
# FUNCIONES PRINCIPALES
# ============================================================================

def login():
    """Realiza el inicio de sesión"""
    imprimir_seccion("🔐 INICIANDO SESIÓN")
    
    try:
        # Clic en botón "Ingresar"
        try:
            boton_ingresar = driver.find_element(By.XPATH, "//a[contains(text(), 'Ingresar')]")
            boton_ingresar.click()
            time.sleep(2)
        except Exception:
            pass
        
        # Cambiar al iframe de login
        driver.switch_to.default_content()
        driver.switch_to.frame(IFRAME_LOGIN)
        
        # Ingresar credenciales
        input_usuario = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[2]/input"))
        )
        input_usuario.send_keys(USUARIO)
        
        input_contrasena = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[3]/input"))
        )
        input_contrasena.send_keys(CONTRASENA)
        
        # Hacer clic en botón de login
        boton_login = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[7]/input"))
        )
        boton_login.click()
        time.sleep(5)
        
        print("✅ Inicio de sesión exitoso")
        
        # Seleccionar rol (opción 4)
        driver.switch_to.default_content()
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="seleccionRol:roles"]/option[4]'))
        ).click()
        time.sleep(4)
        print("✅ Rol de 'Subdirector' (o equivalente) seleccionado.")
        
        return True
        
    except Exception as e:
        print(f"❌ Error durante el login o selección de rol: {e}")
        return False

def navegar_a_finalizar_proceso_y_filtrar():
    """Navega a la sección de Finalizar Proceso de Matrícula y aplica el primer filtro CON FUERZA BRUTA."""
    imprimir_seccion("🧭 NAVEGANDO Y FILTRANDO (CON BÚSQUEDA EXHAUSTIVA)")
    
    try:
        # 1. Navegar por el menú lateral
        driver.switch_to.default_content()
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, XPATH_MENU_MATRICULA))
        ).click()
        
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, XPATH_OPCION_PROCESO_MATRICULA))
        ).click()
        
        # 2. Clic en "Finalizar el proceso de matrícula"
        elemento_finalizar = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, XPATH_FINALIZAR_PROCESO))
        )
        driver.execute_script("arguments[0].click();", elemento_finalizar)
        time.sleep(5)  # Espera más tiempo para asegurar la carga completa
        print("✅ Acceso a 'Finalizar el proceso de matrícula' exitoso.")
        
        # 3. LISTAR TODOS LOS IFRAMES DISPONIBLES
        listar_todos_los_iframes()
        
        # 4. BÚSQUEDA EXHAUSTIVA DEL SELECT
        opciones_esperadas = ["COMPLEMENTARIA", "Complementaria", "complementaria"]
        encontrado, iframe_idx, select_elem = buscar_select_en_todos_los_iframes(
            ID_SELECT_TIPO_FORMACION, 
            opciones_esperadas
        )
        
        if not encontrado:
            # Si no lo encontramos por ID, intentar buscar CUALQUIER select
            print("\n⚠️ Intentando buscar CUALQUIER <select> en los iframes...")
            encontrado_alt, iframe_idx_alt = buscar_cualquier_select()
            if not encontrado_alt:
                raise Exception(f"❌ No se encontró el select '{ID_SELECT_TIPO_FORMACION}' en ningún iframe")
            else:
                raise Exception(f"❌ Se encontraron selects pero no el ID '{ID_SELECT_TIPO_FORMACION}'")
        
        # 5. SELECCIONAR LA OPCIÓN
        if not seleccionar_opcion_en_select(select_elem, opciones_esperadas):
            raise Exception("❌ No se pudo seleccionar la opción COMPLEMENTARIA")
        
        # 6. Buscar y hacer clic en el botón "Buscar"
        # Mantener el foco en el mismo iframe donde encontramos el select
        time.sleep(1)
        
        try:
            boton_buscar = driver.find_element(By.XPATH, XPATH_BOTON_BUSCAR_FICHA)
            boton_buscar.click()
            print("✅ Búsqueda inicial de fichas ejecutada con éxito.")
        except NoSuchElementException:
            # Si no se encuentra por XPath, buscar por otros métodos
            print("⚠️ Botón 'Buscar' no encontrado por XPath, buscando alternativas...")
            botones = driver.find_elements(By.TAG_NAME, "button")
            botones.extend(driver.find_elements(By.TAG_NAME, "input"))
            
            for btn in botones:
                btn_type = btn.get_attribute("type")
                btn_value = btn.get_attribute("value")
                btn_text = btn.text
                
                if btn_type in ["submit", "button"] and (
                    "buscar" in btn_value.lower() or 
                    "buscar" in btn_text.lower() or
                    btn.get_attribute("id") == "frmFichas:buscarFichas"
                ):
                    btn.click()
                    print(f"✅ Clic en botón alternativo: {btn_value or btn_text}")
                    break
        
        time.sleep(5)
        return True
        
    except Exception as e:
        print(f"❌ Error durante la navegación y filtro: {e}")
        driver.switch_to.default_content()
        return False

def abrir_modal_filtros():
    """Abre el modal de filtros haciendo clic en el ícono"""
    imprimir_seccion("🔍 ABRIENDO MODAL DE FILTROS")
    
    try:
        # Esperar a que termine de cargar la tabla de resultados
        time.sleep(3)
        
        # Primero intentar en el iframe conocido
        driver.switch_to.default_content()
        
        try:
            WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, IFRAME_CONTENIDO)))
            icono_filtros = driver.find_element(By.XPATH, XPATH_ICONO_FILTROS)
            driver.execute_script("arguments[0].click();", icono_filtros)
            print("✅ Clic en ícono de filtros ejecutado (XPath original)")
            time.sleep(8)
            return True
        except Exception as e_xpath:
            print(f"⚠️ XPath original falló: {str(e_xpath)[:80]}")
            print("⚠️ Buscando ícono de filtros exhaustivamente...")
            
            # Búsqueda exhaustiva en el iframe 'contenido'
            driver.switch_to.default_content()
            WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, IFRAME_CONTENIDO)))
            
            # Estrategia 1: Buscar todos los <img>
            print("\n   🔎 Estrategia 1: Buscando imágenes...")
            iconos_img = driver.find_elements(By.TAG_NAME, "img")
            print(f"   📊 Total de <img> encontrados: {len(iconos_img)}")
            
            for idx, icono in enumerate(iconos_img):
                try:
                    src = icono.get_attribute("src") or ""
                    alt = icono.get_attribute("alt") or ""
                    title = icono.get_attribute("title") or ""
                    onclick = icono.get_attribute("onclick") or ""
                    
                    print(f"      IMG[{idx}]: src='{src[-30:]}', alt='{alt}', title='{title}', onclick='{onclick[:30]}'")
                    
                    # Buscar palabras clave relacionadas con filtros
                    if any(keyword in src.lower() for keyword in ["filter", "filtro", "search", "buscar", "lupa", "magnif"]) or \
                       any(keyword in alt.lower() for keyword in ["filter", "filtro", "search", "buscar", "lupa"]) or \
                       any(keyword in title.lower() for keyword in ["filter", "filtro", "search", "buscar"]) or \
                       any(keyword in onclick.lower() for keyword in ["filter", "filtro", "modal", "dialog"]):
                        try:
                            driver.execute_script("arguments[0].click();", icono)
                            print(f"   ✅ Clic exitoso en IMG[{idx}]")
                            time.sleep(8)
                            return True
                        except:
                            print(f"   ⚠️ Clic falló en IMG[{idx}]")
                except:
                    continue
            
            # Estrategia 2: Buscar todos los <a> (enlaces con iconos)
            print("\n   🔎 Estrategia 2: Buscando enlaces con iconos...")
            enlaces = driver.find_elements(By.TAG_NAME, "a")
            print(f"   📊 Total de <a> encontrados: {len(enlaces)}")
            
            for idx, enlace in enumerate(enlaces):
                try:
                    href = enlace.get_attribute("href") or ""
                    onclick = enlace.get_attribute("onclick") or ""
                    title = enlace.get_attribute("title") or ""
                    texto = enlace.text or ""
                    
                    # Buscar enlaces que contengan imágenes dentro
                    imgs_dentro = enlace.find_elements(By.TAG_NAME, "img")
                    
                    if imgs_dentro and (
                        any(keyword in onclick.lower() for keyword in ["filter", "filtro", "modal", "dialog"]) or
                        any(keyword in title.lower() for keyword in ["filter", "filtro", "buscar"]) or
                        any(keyword in href.lower() for keyword in ["filter", "filtro", "modal"])
                    ):
                        print(f"      ENLACE[{idx}]: {len(imgs_dentro)} img(s), onclick='{onclick[:40]}', title='{title}'")
                        try:
                            driver.execute_script("arguments[0].click();", enlace)
                            print(f"   ✅ Clic exitoso en ENLACE[{idx}]")
                            time.sleep(8)
                            return True
                        except:
                            print(f"   ⚠️ Clic falló en ENLACE[{idx}]")
                except:
                    continue
            
            # Estrategia 3: Buscar por clases CSS comunes
            print("\n   🔎 Estrategia 3: Buscando por clases CSS...")
            selectores_css = [
                "a[title*='filtr']", "a[title*='Filtr']",
                "img[alt*='filtr']", "img[alt*='Filtr']",
                "a.filter", "a.filtro", "button.filter", "button.filtro",
                "a[onclick*='modal']", "a[onclick*='dialog']"
            ]
            
            for selector in selectores_css:
                try:
                    elementos = driver.find_elements(By.CSS_SELECTOR, selector)
                    if elementos:
                        print(f"      ✅ Encontrado con selector: {selector} ({len(elementos)} elemento(s))")
                        driver.execute_script("arguments[0].click();", elementos[0])
                        print(f"   ✅ Clic exitoso con selector CSS")
                        time.sleep(8)
                        return True
                except:
                    continue
            
            raise Exception("No se encontró el ícono de filtros después de búsqueda exhaustiva")
        
    except Exception as e:
        print(f"❌ Error al abrir modal de filtros: {e}")
        return False

def ingresar_ficha_y_buscar(numero_ficha):
    """Ingresa el número de ficha en el modal y ejecuta la búsqueda"""
    imprimir_seccion(f"📝 INGRESANDO FICHA: {numero_ficha}")
    
    try:
        driver.switch_to.default_content()
        
        # Buscar estructura de iframes anidados
        WebDriverWait(driver, 10).until( 
            EC.frame_to_be_available_and_switch_to_it((By.ID, IFRAME_CONTENIDO))
        )
        
        WebDriverWait(driver, 10).until(
            EC.frame_to_be_available_and_switch_to_it((By.ID, IFRAME_MODAL))
        )
        print(f"✅ Foco en iframe modal: '{IFRAME_MODAL}'")
        
        input_ficha = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, XPATH_INPUT_FICHA))
        )
        input_ficha.clear()
        input_ficha.send_keys(numero_ficha)
        print(f"✅ Ficha '{numero_ficha}' ingresada correctamente")
        
        boton_buscar = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, XPATH_BOTON_BUSCAR_MODAL))
        )
        boton_buscar.click()
        print("✅ Búsqueda ejecutada con éxito")
        
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"❌ Error al ingresar ficha o ejecutar búsqueda: {e}")
        return False

def main():
    """Función principal que ejecuta todo el flujo"""
    imprimir_seccion("🚀 INICIANDO AUTOMATIZACIÓN SENA SOFIA PLUS")
    
    try:
        driver.get(URL)
        time.sleep(3)
        print(f"✅ Página cargada: {URL}")
        
        if not login():
            raise Exception("Error en login")
        
        if not navegar_a_finalizar_proceso_y_filtrar():
            raise Exception("Error en navegación y filtro inicial")
        
        if not abrir_modal_filtros():
            raise Exception("Error al abrir modal")
        
        if not ingresar_ficha_y_buscar(FICHA_A_INGRESAR):
            raise Exception("Error al buscar ficha")
        
        imprimir_seccion("✅✅✅ PROCESO COMPLETADO CON ÉXITO ✅✅✅")
        print(f"   Ficha '{FICHA_A_INGRESAR}' procesada correctamente")
        
        print("\n⏸️ Navegador permanecerá abierto por 10 segundos...")
        time.sleep(10)
        
    except Exception as e:
        imprimir_seccion("❌ ERROR CRÍTICO")
        print(f"   {str(e)}")
        print("\n⏸️ Navegador permanecerá abierto por 15 segundos para inspección...")
        time.sleep(15)
        
    finally:
        driver.quit()
        print("\n🔚 Navegador cerrado. Proceso finalizado.")

# ============================================================================
# EJECUCIÓN
# ============================================================================

if __name__ == "__main__":
    main()


  🚀 INICIANDO AUTOMATIZACIÓN SENA SOFIA PLUS
✅ Página cargada: http://senasofiaplus.edu.co/sofia-public/

  🔐 INICIANDO SESIÓN
✅ Inicio de sesión exitoso
✅ Rol de 'Subdirector' (o equivalente) seleccionado.

  🧭 NAVEGANDO Y FILTRANDO (CON BÚSQUEDA EXHAUSTIVA)
✅ Acceso a 'Finalizar el proceso de matrícula' exitoso.
📊 Total de iframes encontrados: 5
   [0] ID: 'contenido' | NAME: 'contenido' | SRC: 'http://senasofiaplus.edu.co/sofia/staticContent/ht...'
   [1] ID: 'ifraModal' | NAME: 'SIN NAME' | SRC: 'http://senasofiaplus.edu.co/sofia/home/principal.f...'
   [2] ID: 'ifraModal2' | NAME: 'SIN NAME' | SRC: 'http://senasofiaplus.edu.co/sofia/home/principal.f...'
   [3] ID: 'ag-frame' | NAME: 'ag-frame' | SRC: 'http://botformacion.sena.edu.co/agenti_lite_senafo...'
   [4] ID: 'ag-frame' | NAME: 'ag-frame' | SRC: 'http://botformacion.sena.edu.co/agenti_lite_senafo...'

  🔍 BÚSQUEDA DE SELECT 'tipoFormacion' EN TODOS LOS IFRAMES
📊 Explorando 5 iframes...

   🔎 Explorando iframe [0]: ID='cont